# HW3 — Binary-entropy bracket verifier from scratch

Goal: build `hbin_inverse`, `upper`, `lower`, a random verifier of $\mathrm{lower}(H) \le \varepsilon \le \mathrm{upper}(H)$, and grid-search $(\varepsilon^*, w^*) = (1/5, 0.1610)$.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw3', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from math import log2
def hbin(p: float) -> float:
    return 0.0 if (p <= 0 or p >= 1) else -(p*log2(p) + (1-p)*log2(1-p))


## Q1 — `hbin_inverse(h)` via bisection on $[0, 1/2]$.

$H_\mathrm{bin}$ is strictly increasing on $[0, 1/2]$, so its inverse on that branch is well-defined.

In [ ]:
def hbin_inverse(h: float, tol: float = 1e-12) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Closed-form inverse is impossible (transcendental); we use bisection.

In [ ]:
for eps in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    print(f'  eps={eps:.2f} -> H={hbin(eps):.6f} -> H^{{-1}}(H)={hbin_inverse(hbin(eps)):.6f}')


**Gate Q1.** Round-trip error $< 10^{-9}$ on the grid.

In [ ]:
for eps in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    rt = hbin_inverse(hbin(eps))
    assert abs(rt - eps) < 1e-9, f'Q1: round-trip fails at eps={eps}: {rt}'
print('[GATE OK] Q1: hbin_inverse round-trips to 1e-9 on six grid points')


In [ ]:
reflect.log('Q3.Q1_hbin_inverse', 'bisection inverse round-trips ≤1e-9', 'HIGH')


## Q2 — `upper(h) = h/2`, `lower(h) = H^{-1}(h)`, envelope plot.

**Concept.** The bracket has width 0 at $h=0$ and at $h=1$; widest near $h \approx H_\mathrm{bin}(1/5) \approx 0.7219$.

In [ ]:
def upper(h: float) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Upper is linear; lower is concave-then-convex inverse. Their gap is $w(h) = h/2 - H_\mathrm{bin}^{-1}(h)$.

In [ ]:
hs = np.linspace(0, 1, 401)
ups = np.array([upper(h) for h in hs])
los = np.array([lower(h) for h in hs])
fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(hs, los, ups, color='C2', alpha=0.25, label='slack region')
ax.plot(hs, ups, label='upper = h/2', lw=2)
ax.plot(hs, los, label='lower = $H_{bin}^{-1}(h)$', lw=2)
ax.set_xlabel('h = H(Y|Π)'); ax.set_ylabel('ε'); ax.legend()
ax.set_title('Q2 — bracket envelope and slack')
plt.tight_layout()
_plots = _PROJECTS / 'psets' / 'hw3' / 'plots'; _plots.mkdir(exist_ok=True)
fig.savefig(_plots / 'hw3_q2_envelope.png', dpi=120); plt.show()


**Gate Q2.** lower ≤ upper everywhere; widest near $h \approx 0.72$ (corresponding to $\varepsilon^* = 1/5$).

In [ ]:
assert np.all(los <= ups + 1e-12), 'Q2: lower > upper somewhere — bug'
gap = ups - los
h_argmax = hs[int(np.argmax(gap))]
assert 0.65 < h_argmax < 0.78, f'Q2: argmax h={h_argmax} not in expected band near 0.72'
assert gap.max() > 0.10, f'Q2: max gap={gap.max()} too small'
print(f'[GATE OK] Q2: lower ≤ upper; max gap {gap.max():.4f} at h≈{h_argmax:.3f}')


In [ ]:
reflect.log('Q3.Q2_bracket', 'lower ≤ upper; widest near h≈0.72', 'HIGH')


## Q3 — Random-sample verifier of $\mathrm{lower}(H) \le \varepsilon \le \mathrm{upper}(H)$.

Sample cell masses via Multinomial; sample per-cell errors uniformly in $[0, 1/2]$ (Bayes errors live there).

In [ ]:
def random_partition_stats(rng, m: int, n: int):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Sampling errors $> 1/2$ would falsely violate; clamping to $[0, 1/2]$ is essential.

In [ ]:
ok = verify_bracket(num_samples=2000)
print(f'verify_bracket(2000) -> {ok}')


**Gate Q3.** `verify_bracket` returns True on 2000 samples.

In [ ]:
assert ok is True, 'Q3: verifier did not return True'
print('[GATE OK] Q3: 2000 random partitions all in bracket')


In [ ]:
reflect.log('Q3.Q3_verifier', 'bracket holds on 2000 random multinomial+uniform partitions', 'HIGH')


## Q4 — Grid search $\varepsilon^*, w^*$.

Maximise $w(\varepsilon) = H_\mathrm{bin}(\varepsilon)/2 - \varepsilon$ on $[0, 1/2]$.

In [ ]:
def find_w_star(grid: int = 5000):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — false lead.** The slack maximum is NOT at $\varepsilon = 1/2$ (the entropy maximum); first-order condition gives $H_\mathrm{bin}'(\varepsilon) = 2 \Rightarrow \varepsilon^* = 1/5$.

In [ ]:
eps_star, w_star = find_w_star(5000)
print(f'eps* = {eps_star:.4f} (expected 0.2000)')
print(f'w*   = {w_star:.4f} (expected 0.1610)')


**Gate Q4.** $\varepsilon^* \in (0.199, 0.201)$ and $w^* \in (0.160, 0.162)$.

In [ ]:
assert 0.199 < eps_star < 0.201, f'Q4: eps*={eps_star} out of band'
assert 0.160 < w_star  < 0.162, f'Q4: w*={w_star} out of band'
print(f'[GATE OK] Q4: eps*={eps_star:.4f}, w*={w_star:.4f}')


In [ ]:
reflect.log('Q3.Q4_wstar', f'eps*={eps_star:.4f}, w*={w_star:.4f}', 'HIGH')


## Q5 — Writeup & calibration.

In [ ]:
reflect.log('Q3.Q5_writeup', 'four sections + five calibration entries; bracket holds end-to-end', 'MEDIUM')
print('HW3 reflection log:')
for r in reflect.dump():
    if 'hw3' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW3.**